In [63]:
import warnings
import yfinance as yf
warnings.filterwarnings("ignore")

In [64]:
import pandas as pd
import numpy as np

options_df = pd.read_csv("merged_options_data.csv")


# Create a unique list of daily stock prices from your data
daily_spot = options_df[['Date', 'Underlying Value']].drop_duplicates().sort_values('Date')
daily_spot['Underlying Value']=pd.to_numeric(daily_spot['Underlying Value'],errors='coerce')

# Calculate Log Returns
daily_spot['returns'] = np.log(daily_spot['Underlying Value'] / daily_spot['Underlying Value'].shift(1))

# 20-Day Annualized Volatility (Standard "Sigma" for Black-Scholes)
daily_spot['sigma'] = daily_spot['returns'].rolling(window=20).std() * np.sqrt(252)
# daily_spot['sigma']= daily_spot['sigma'].shift(1)


# Merge this 'sigma' back into the main options dataframe
options_df = pd.merge(options_df, daily_spot[['Date','sigma']], on='Date', how='left')

# Drop the first 20 days since they won't have a sigma value
options_df = options_df.dropna(subset=['sigma'])

In [65]:
daily_spot

,Date,Underlying Value,returns,sigma
0,2023-01-02,2876.6,NaN,NaN
173,2023-01-03,2876.6,0.000000,NaN
355,2023-01-04,2876.6,0.000000,NaN
537,2023-01-05,2876.6,0.000000,NaN
722,2023-01-06,2876.6,0.000000,NaN
...,...,...,...,...
100277,2025-12-24,2785.5,-0.007974,0.222313
100386,2025-12-26,2746.5,-0.014100,0.226258
100495,2025-12-29,2775.4,0.010468,0.230803
100605,2025-12-30,2758.3,-0.006180,0.231334


In [66]:
# Inputs for your future model
final_dataset = pd.DataFrame({
    'Date': options_df['Date'],
    'S': options_df['Underlying Value'],
    'K': options_df['Strike Price'],
    'T': options_df['T'],
    'r': 0.067, # Average risk-free rate for India 2025
    'sigma': options_df['sigma'],
    'Market_Price': options_df['Settle Price'] # This is what we want to predict
})

final_dataset["Market_Price"] = pd.to_numeric(final_dataset["Market_Price"], errors='coerce')

print("Dataset prepared for Asian Paints!")
print(final_dataset.head())

# final_dataset.to_csv("AsianPaints_Model_Data_2023_25.csv", index=False)

Dataset prepared for Asian Paints!
            Date       S       K         T      r  sigma  Market_Price
3759  2023-01-31  2876.6  2820.0  0.156164  0.067    0.0        103.15
3760  2023-01-31  2876.6  3240.0  0.156164  0.067    0.0         14.85
3761  2023-01-31  2876.6  3460.0  0.156164  0.067    0.0          4.25
3762  2023-01-31  2876.6  3040.0  0.156164  0.067    0.0         40.50
3763  2023-01-31  2876.6  2620.0  0.156164  0.067    0.0        206.00


In [67]:
vix_data=pd.read_csv("VIX_data.csv")
vix_data.drop('Unnamed: 0', axis=1, inplace=True)
vix_data['Date'] = pd.to_datetime(vix_data['Date']).dt.tz_localize(None)
final_dataset['Date'] = pd.to_datetime(final_dataset['Date']).dt.tz_localize(None)

final_dataset = final_dataset.merge(vix_data, on="Date", how="left")

final_dataset['vix'] = final_dataset['vix'].ffill()

print(final_dataset[['Date', 'S', 'vix']].head())



        Date       S        vix
0 2023-01-31  2876.6  16.879999
1 2023-01-31  2876.6  16.879999
2 2023-01-31  2876.6  16.879999
3 2023-01-31  2876.6  16.879999
4 2023-01-31  2876.6  16.879999


In [68]:
stock_data=pd.read_csv("stock_data_2023_25.csv")
stock_data.drop('Unnamed: 0', axis=1, inplace=True)
stock_data['Date'] = pd.to_datetime(stock_data['Date']).dt.tz_localize(None)

In [69]:
# Calculate 5-day rolling average
stock_data['vol_ma'] = stock_data['Volume'].rolling(window=5).mean()

# Merge into your dataset
final_dataset = final_dataset.merge(stock_data[['Date', 'vol_ma']], on="Date", how="left")
final_dataset['vol_ma'] = final_dataset['vol_ma'].ffill()

In [70]:
cols_to_convert = ["S", "K", "T", "Market_Price", "sigma"]

for col in cols_to_convert:
    final_dataset[col] = pd.to_numeric(final_dataset[col], errors='coerce')

In [71]:
final_dataset

,Date,S,K,T,r,sigma,Market_Price,vix,vol_ma
0,2023-01-31,2876.6,2820.0,0.156164,0.067,0.000000,103.15,16.879999,1370927.8
1,2023-01-31,2876.6,3240.0,0.156164,0.067,0.000000,14.85,16.879999,1370927.8
2,2023-01-31,2876.6,3460.0,0.156164,0.067,0.000000,4.25,16.879999,1370927.8
3,2023-01-31,2876.6,3040.0,0.156164,0.067,0.000000,40.50,16.879999,1370927.8
4,2023-01-31,2876.6,2620.0,0.156164,0.067,0.000000,206.00,16.879999,1370927.8
...,...,...,...,...,...,...,...,...,...
77179,2025-12-31,2769.5,2600.0,0.243836,0.067,0.200426,240.60,9.480000,1593596.6
77180,2025-12-31,2769.5,2640.0,0.243836,0.067,0.200426,211.75,9.480000,1593596.6
77181,2025-12-31,2769.5,2800.0,0.243836,0.067,0.200426,117.10,9.480000,1593596.6
77182,2025-12-31,2769.5,2960.0,0.243836,0.067,0.200426,56.55,9.480000,1593596.6


In [72]:
final_dataset.to_csv("AsianPaints_Model_Data_2023_25.csv", index=False)